# FastAPI Introduction

## What is FastAPI?

FastAPI is a **modern, fast (high-performance)** web framework for building APIs with Python based on standard Python type hints.

**Key Features:**
- Fast: Very high performance, on par with NodeJS and Go
- Fast to code: Increases developer speed by ~200-300%
- Fewer bugs: Reduces human-induced errors by ~40%
- Intuitive: Great editor support with auto-completion
- Easy: Designed to be easy to use and learn
- Short: Minimizes code duplication
- Robust: Production-ready code with automatic interactive documentation
- Standards-based: Based on OpenAPI and JSON Schema

---

## ASGI vs WSGI

| Feature | WSGI | ASGI |
|---------|------|------|
| Stands for | Web Server Gateway Interface | Asynchronous Server Gateway Interface |
| Concurrency | Synchronous (one request at a time per worker) | Asynchronous (handles many concurrent requests) |
| WebSockets | ❌ Not supported | ✅ Supported |
| HTTP/2 | ❌ Limited | ✅ Supported |
| Frameworks | Flask, Django | FastAPI, Starlette, Django Channels |
| Server | Gunicorn, uWSGI | Uvicorn, Hypercorn, Daphne |

FastAPI is built on **Starlette** (ASGI) and **Pydantic** (data validation).

```
FastAPI
  └── Starlette (ASGI framework)
        └── Uvicorn (ASGI server)
  └── Pydantic (data validation)
```

In [1]:
# Install FastAPI
# pip install fastapi uvicorn[standard] python-multipart

# Verify installation
import fastapi
print(f"FastAPI version: {fastapi.__version__}")

FastAPI version: 0.137.2


## First FastAPI Application

In [2]:
# first_app.py save and run with: uvicorn first_app:app --reload

from fastapi import FastAPI

app = FastAPI(
    title="My First API",
    description="A comprehensive FastAPI example",
    version="1.0.0"
)

@app.get("/")
def read_root():
    return {"message": "Hello, World!"}

@app.get("/health")
def health_check():
    return {"status": "healthy"}

# Run: uvicorn first_app:app --reload --port 8000
# Docs: http://localhost:8000/docs
# ReDoc: http://localhost:8000/redoc
# OpenAPI JSON: http://localhost:8000/openapi.json

## HTTP Methods

| Method | Purpose | Idempotent | Safe |
|--------|---------|-----------|------|
| GET | Retrieve resource | ✅ | ✅ |
| POST | Create resource | ❌ | ❌ |
| PUT | Replace resource (full update) | ✅ | ❌ |
| PATCH | Partial update | ❌ | ❌ |
| DELETE | Delete resource | ✅ | ❌ |
| HEAD | Like GET but no body | ✅ | ✅ |
| OPTIONS | Get allowed methods | ✅ | ✅ |

In [3]:
from fastapi import FastAPI, Path, Query, Body, status
from pydantic import BaseModel, Field
from typing import Optional, List

app = FastAPI()

# --- In-memory database ---
items_db = {}

# --- Pydantic Models ---
class ItemCreate(BaseModel):
    name: str = Field(..., min_length=1, max_length=100, description="Item name")
    price: float = Field(..., gt=0, description="Price must be positive")
    description: Optional[str] = Field(None, max_length=500)
    tags: List[str] = []

class ItemResponse(ItemCreate):
    id: int

    class Config:
        from_attributes = True

# --- Path Parameters ---
@app.get("/items/{item_id}", response_model=ItemResponse)
def get_item(
    item_id: int = Path(..., ge=1, description="The ID of the item")
):
    """Get a specific item by ID."""
    if item_id not in items_db:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail=f"Item {item_id} not found")
    return items_db[item_id]

# --- Query Parameters ---
@app.get("/items/", response_model=List[ItemResponse])
def list_items(
    skip: int = Query(0, ge=0, description="Items to skip"),
    limit: int = Query(10, ge=1, le=100, description="Max items to return"),
    search: Optional[str] = Query(None, description="Search in name")
):
    """List all items with optional filtering."""
    items = list(items_db.values())
    if search:
        items = [i for i in items if search.lower() in i["name"].lower()]
    return items[skip: skip + limit]

# --- Request Body (POST) ---
@app.post("/items/", response_model=ItemResponse, status_code=status.HTTP_201_CREATED)
def create_item(item: ItemCreate):
    """Create a new item."""
    item_id = len(items_db) + 1
    new_item = {"id": item_id, **item.model_dump()}
    items_db[item_id] = new_item
    return new_item

# --- PUT (full update) ---
@app.put("/items/{item_id}", response_model=ItemResponse)
def update_item(item_id: int, item: ItemCreate):
    """Fully replace an item."""
    if item_id not in items_db:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail="Item not found")
    items_db[item_id] = {"id": item_id, **item.model_dump()}
    return items_db[item_id]

# --- PATCH (partial update) ---
@app.patch("/items/{item_id}")
def partial_update_item(item_id: int, item: ItemCreate):
    stored = items_db.get(item_id)
    if not stored:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail="Item not found")
    update_data = item.model_dump(exclude_unset=True)
    stored.update(update_data)
    return stored

# --- DELETE ---
@app.delete("/items/{item_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_item(item_id: int):
    """Delete an item."""
    if item_id not in items_db:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail="Item not found")
    del items_db[item_id]

/tmp/ipykernel_158653/2462928713.py:17: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class ItemResponse(ItemCreate):


## Pydantic Models Data Validation

Pydantic validates data at runtime using Python type hints.

In [4]:
from pydantic import BaseModel, Field, EmailStr, validator, model_validator
from typing import Optional, List
from datetime import datetime
from enum import Enum

class UserRole(str, Enum):
    admin = "admin"
    user = "user"
    guest = "guest"

class Address(BaseModel):
    street: str
    city: str
    country: str = "US"
    zip_code: str

class UserCreate(BaseModel):
    username: str = Field(..., min_length=3, max_length=50, pattern=r'^[a-zA-Z0-9_]+$')
    email: str  # Use EmailStr with: pip install pydantic[email]
    age: Optional[int] = Field(None, ge=0, le=150)
    role: UserRole = UserRole.user
    address: Optional[Address] = None
    tags: List[str] = []

    @validator('username')
    def username_not_reserved(cls, v):
        reserved = ['admin', 'root', 'system']
        if v.lower() in reserved:
            raise ValueError(f'{v} is a reserved username')
        return v

class UserResponse(BaseModel):
    id: int
    username: str
    email: str
    role: UserRole
    created_at: datetime

    class Config:
        from_attributes = True

# Test validation
try:
    user = UserCreate(username="john_doe", email="john@example.com", age=25)
    print(user.model_dump())
except Exception as e:
    print(f"Validation error: {e}")

{'username': 'john_doe', 'email': 'john@example.com', 'age': 25, 'role': <UserRole.user: 'user'>, 'address': None, 'tags': []}


/tmp/ipykernel_158653/3616391060.py:25: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator('username')
/tmp/ipykernel_158653/3616391060.py:32: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class UserResponse(BaseModel):


## Status Codes

| Code | Category | Common Codes |
|------|----------|--------------|
| 1xx | Informational | 100 Continue |
| 2xx | Success | 200 OK, 201 Created, 204 No Content |
| 3xx | Redirection | 301 Moved Permanently, 307 Temporary Redirect |
| 4xx | Client Error | 400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found, 422 Unprocessable Entity |
| 5xx | Server Error | 500 Internal Server Error, 503 Service Unavailable |

In [5]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError

app = FastAPI()

# Custom exception class
class ItemNotFoundException(Exception):
    def __init__(self, item_id: int):
        self.item_id = item_id

# Custom exception handler
@app.exception_handler(ItemNotFoundException)
async def item_not_found_handler(request: Request, exc: ItemNotFoundException):
    return JSONResponse(
        status_code=404,
        content={"detail": f"Item {exc.item_id} was not found", "type": "item_not_found"}
    )

# Override validation error handler
@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={
            "detail": exc.errors(),
            "body": str(exc.body)
        }
    )

@app.get("/items/{item_id}")
async def get_item(item_id: int):
    if item_id == 0:
        raise ItemNotFoundException(item_id=item_id)
    if item_id < 0:
        raise HTTPException(
            status_code=400,
            detail="Item ID must be positive",
            headers={"X-Error": "InvalidID"}
        )
    return {"item_id": item_id}

## Headers and Cookies

In [6]:
from fastapi import FastAPI, Header, Cookie, Response
from typing import Optional

app = FastAPI()

@app.get("/headers")
async def read_headers(
    user_agent: Optional[str] = Header(None),
    x_token: Optional[str] = Header(None),
    accept_language: Optional[str] = Header(None)
):
    return {
        "user_agent": user_agent,
        "x_token": x_token,
        "accept_language": accept_language
    }

@app.get("/cookies")
async def read_cookies(
    session_id: Optional[str] = Cookie(None),
    user_preference: Optional[str] = Cookie(None)
):
    return {"session_id": session_id, "user_preference": user_preference}

@app.post("/set-cookie")
async def set_cookie(response: Response):
    response.set_cookie(
        key="session_id",
        value="abc123",
        max_age=3600,
        httponly=True,
        samesite="lax"
    )
    return {"message": "Cookie set"}

@app.post("/set-headers")
async def set_response_headers(response: Response):
    response.headers["X-Custom-Header"] = "MyValue"
    response.headers["X-Process-Time"] = "0.05"
    return {"message": "Headers set"}

## File Uploads

In [7]:
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
from typing import List
import shutil
import os

app = FastAPI()
UPLOAD_DIR = "uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# Single file upload
@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    file_path = f"{UPLOAD_DIR}/{file.filename}"
    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    return {
        "filename": file.filename,
        "content_type": file.content_type,
        "size": os.path.getsize(file_path)
    }

# Multiple files
@app.post("/upload-multiple")
async def upload_multiple(files: List[UploadFile] = File(...)):
    results = []
    for file in files:
        content = await file.read()
        results.append({"filename": file.filename, "size": len(content)})
    return results

# Form data + file
@app.post("/upload-with-form")
async def upload_with_form(
    file: UploadFile = File(...),
    description: str = Form(...),
    category: str = Form("general")
):
    return {
        "filename": file.filename,
        "description": description,
        "category": category
    }

# File download
@app.get("/download/{filename}")
async def download_file(filename: str):
    file_path = f"{UPLOAD_DIR}/{filename}"
    if not os.path.exists(file_path):
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(path=file_path, filename=filename)

## Automatic API Documentation

FastAPI automatically generates interactive documentation:

- **Swagger UI**: `http://localhost:8000/docs`
- **ReDoc**: `http://localhost:8000/redoc`
- **OpenAPI JSON**: `http://localhost:8000/openapi.json`

Customize the docs:

In [8]:
from fastapi import FastAPI
from fastapi.openapi.utils import get_openapi

app = FastAPI(
    title="AI Learning API",
    description="""
    ## AI Learning Platform API
    
    This API provides endpoints for:
    * **ML Model Inference** - Run predictions
    * **Dataset Management** - Upload and manage datasets
    * **User Management** - Authentication and profiles
    """,
    version="2.0.0",
    terms_of_service="https://example.com/terms",
    contact={"name": "Support", "email": "support@example.com"},
    license_info={"name": "MIT"},
    docs_url="/api/docs",     # Change docs URL
    redoc_url="/api/redoc",   # Change redoc URL
    openapi_url="/api/openapi.json"
)

# Tag items for organized docs
@app.get("/users", tags=["users"], summary="List all users", response_description="List of users")
async def list_users():
    """Retrieve all users with their profiles.
    
    Returns a list of user objects containing:
    - **id**: unique identifier
    - **username**: user's handle
    - **email**: user's email
    """
    return []

@app.get("/predictions", tags=["ml"])
async def list_predictions():
    return []

print("API docs available at: /api/docs")

API docs available at: /api/docs


## Additional Learning Resources

### Official Documentation
- [FastAPI Official Docs](https://fastapi.tiangolo.com/) Best docs in any framework
- [Pydantic V2 Docs](https://docs.pydantic.dev/latest/) Data validation library
- [Starlette Docs](https://www.starlette.io/) ASGI framework FastAPI is built on
- [Uvicorn Docs](https://www.uvicorn.org/) ASGI server

### Books
- [FastAPI by Packt](https://www.packtpub.com/product/building-data-science-applications-with-fastapi/9781801079211) Building Data Science Applications with FastAPI

### Video Courses
- [FastAPI Full Course - freeCodeCamp](https://www.youtube.com/watch?v=7t2alSnE2-I) Free comprehensive course
- [FastAPI Tutorial - Tech with Tim](https://www.youtube.com/watch?v=tLKKmouUams) Quick start guide

### GitHub Repositories
- [tiangolo/fastapi](https://github.com/tiangolo/fastapi) Official FastAPI repo
- [fastapi/full-stack-fastapi-template](https://github.com/fastapi/full-stack-fastapi-template) Production template
- [zhanymkanov/fastapi-best-practices](https://github.com/zhanymkanov/fastapi-best-practices) Best practices guide

### Related Tools
- [httpx](https://www.python-httpx.org/) Async HTTP client (for testing)
- [pytest-asyncio](https://pytest-asyncio.readthedocs.io/) Async testing
- [SQLModel](https://sqlmodel.tiangolo.com/) SQL + Pydantic by FastAPI author